# M4 — 근거 평가·조건부 재검색 루프 + 실DB GraphPath (11/06)

일정표 M4 합격 기준: **실제 GraphPath 1개 + 필요시 추가 검색·정상 종료 1개**. 주승우 제출물: 메인 Graph의 **1 Tool 종료 + 추가 Tool 호출** 각각 실행 로그.

재검색 규칙(`src/rag/school/main_graph.py` `evaluate_evidence`):
- 근거가 충분하면(ANSWERED) **추가 호출 없이 종료**.
- 부족할 때만 **명시적 신호**가 가리키는, 아직 부르지 않은 Tool을 부른다. 신호는 ① 질문 분류의 보조 의도 ② Tool 경고의 `[미해결] … <tool 이름>` 두 가지뿐.
- 총 3회 한도, 같은 Tool 재호출 없음. 두 번째 호출부터 앞선 Finding의 질병·개념 ID를 `context_entity_ids`로 넘긴다.
- 주 의도 Tool이 근거를 못 냈는데 다른 Tool만 답하면 PARTIAL + 남은 확인 사항으로 안내한다.

Tool 연결은 M3와 같은 registry 방식(다른 담당자 모듈을 `src`에서 import하지 않음). **legal_search는 고정 샘플 질문 1개만 받는 단계**다.

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)

from index.school.dictionaries import load_dictionaries
from rag.school.action_tool import check_graph_paths
from rag.school.contract_v2_local import validate_query_output
from rag.school.graph import default_deps
from rag.school.main_graph import MAX_TOOL_CALLS, default_registry, run_main
from rag.school.validation_local import verify_against_db
from rag.respiratory.disease_tool import disease_search
from rag.respiratory.crossdoc_tool import crossdoc_graph_search
from rag.prevention_legal.tool import legal_search
from rag.prevention_legal.run import SAMPLE_QUESTION as LEGAL_SAMPLE

deps = default_deps()
deps["books"] = load_dictionaries()
DRIVER, DATABASE = deps["driver"], deps["database"]
ALLOWED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025", "kdca_measles_2026",
           "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521"]
REGISTRY = {**default_registry(deps),
            "disease_search": lambda ti: disease_search(ti, deps=deps),
            "crossdoc_graph_search": lambda ti: crossdoc_graph_search(ti, deps=deps),
            "legal_search": legal_search}
print("연결 Tool:", sorted(REGISTRY), "/ 호출 한도:", MAX_TOOL_CALLS)

연결 Tool: ['action_search', 'crossdoc_graph_search', 'disease_search', 'legal_search'] / 호출 한도: 3


## 1. 재검색 루프 실행 로그 (실DB)

In [2]:
CASES = [
    ("A. 근거 충분 → 1회 종료", "학교에서 유증상 학생이 발견되면 담임교사와 보건교사는 각각 무엇을 확인하나요?"),
    ("B. 보조 의도(질병) → 2회 → 충분 종료", "홍역의 잠복기는 며칠이고, 홍역 학생이 생기면 학교는 무엇을 해야 하나요?"),
    ("C. 법령(샘플) 부분 근거 → 학교 행동 추가 → 2회 종료", LEGAL_SAMPLE),
    ("D. 주 의도(법령) 근거 없음 → 학교 행동만 확보 → PARTIAL 종료",
     "성홍열 학생이 생기면 담임교사는 무엇을 하고, 학교장은 법적으로 보건소에 신고해야 하나요?"),
    ("E. 문서 비교(CrossDoc, GraphPath) → 1회 종료",
     "보호자에게 연락해 진료를 요청하는 일을 교육부 매뉴얼과 경기도 가이드북은 각각 어떻게 설명하나요?"),
]
runs, step_rows = {}, []
for n, (label, question) in enumerate(CASES, 1):
    query = {"question": question, "region": "GYEONGGI", "allowed_document_ids": ALLOWED, "request_id": f"m4-{n:02d}"}
    started = time.perf_counter()
    out = run_main(query, deps, REGISTRY)
    validate_query_output(out, query)
    trace = list(deps["main_trace"])
    db = None
    if out["findings"]:
        report = verify_against_db(out, DRIVER, DATABASE)
        db = "통과" if not report["bad_citations"] and not report["bad_findings"] else str(report)[:80]
    runs[label] = {"query": query, "out": out, "trace": trace, "db": db, "seconds": round(time.perf_counter() - started, 1)}
    for t in trace:
        if t["step"] in ("classify_intent", "select_tool", "execute_tool", "evaluate_evidence"):
            detail = {k: v for k, v in t.items() if k not in ("step", "request_id") and v not in (None, "", [])}
            step_rows.append({"사례": label[:2], "단계": t["step"], "내용": json.dumps(detail, ensure_ascii=False)[:150]})

summary = pd.DataFrame([{"사례": label, "호출 순서": r["out"]["tools_used"], "호출 수": r["out"]["search_rounds"],
                         "최종 status": r["out"]["status"], "Finding": len(r["out"]["findings"]),
                         "Citation": len(r["out"]["citations"]), "DB 재대조": r["db"], "초": r["seconds"]}
                        for label, r in runs.items()])
display(summary)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE any(d IN $docs WHERE node.chunk_id STARTS WITH d + ':'\n                              OR node.chunk_id STARTS WITH 'test_' + d + ':')\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.pdf_page AS pdf_page, score\n  

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE any(d IN $docs WHERE node.chunk_id STARTS WITH d + ':'\n                              OR node.chunk_id STARTS WITH 'test_' + d + ':')\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.pdf_page AS pdf_page, score\n  

,사례,호출 순서,호출 수,최종 status,Finding,Citation,DB 재대조,초
0,A. 근거 충분 → 1회 종료,[action_search],1,ANSWERED,12,11,통과,5.1
1,B. 보조 의도(질병) → 2회 → 충분 종료,"[action_search, disease_search]",2,ANSWERED,18,15,통과,5.9
2,C. 법령(샘플) 부분 근거 → 학교 행동 추가 → 2회 종료,"[legal_search, action_search]",2,PARTIAL,14,11,통과,7.6
3,D. 주 의도(법령) 근거 없음 → 학교 행동만 확보 → PARTIAL 종료,"[legal_search, action_search]",2,PARTIAL,11,11,통과,6.3
4,"E. 문서 비교(CrossDoc, GraphPath) → 1회 종료",[crossdoc_graph_search],1,ANSWERED,11,7,통과,2.4


### 노드별 판단 기록 (select → execute → evaluate의 '재검색/종료' 결정)

In [3]:
display(pd.DataFrame(step_rows))

,사례,단계,내용
0,A.,classify_intent,"{""intent"": ""ACTION""}"
1,A.,select_tool,"{""tool"": ""action_search"", ""registered"": true, ""round"": 1, ""reason"": ""질문의 주 의도""}"
2,A.,execute_tool,"{""tool"": ""action_search"", ""status"": ""OK"", ""findings"": 12, ""citations"": 11, ""graph_paths"": 8, ""seconds"": 4.53}"
3,A.,evaluate_evidence,"{""status"": ""ANSWERED"", ""rounds"": 1, ""unresolved"": 0, ""resolved"": 0, ""decision"": ""종료""}"
4,B.,classify_intent,"{""intent"": ""ACTION"", ""secondary"": [""DISEASE""]}"
5,B.,select_tool,"{""tool"": ""action_search"", ""registered"": true, ""round"": 1, ""reason"": ""질문의 주 의도""}"
6,B.,execute_tool,"{""tool"": ""action_search"", ""status"": ""OK"", ""findings"": 10, ""citations"": 10, ""graph_paths"": 8, ""seconds"": 3.12}"
7,B.,evaluate_evidence,"{""status"": ""PARTIAL"", ""rounds"": 1, ""unresolved"": 0, ""resolved"": 0, ""next_tool"": ""disease_search"", ""decisio..."
8,B.,select_tool,"{""tool"": ""disease_search"", ""registered"": true, ""round"": 2, ""reason"": ""질문의 일부(DISEASE)""}"
9,B.,execute_tool,"{""tool"": ""disease_search"", ""status"": ""OK"", ""findings"": 8, ""citations"": 8, ""graph_paths"": 8, ""seconds"": 1.1}"


### 두 번째 호출에 넘긴 ToolInput (앞선 결과의 질병·개념 ID 전달 확인)

In [4]:
for label in ("B. 보조 의도(질병) → 2회 → 충분 종료", "C. 법령(샘플) 부분 근거 → 학교 행동 추가 → 2회 종료"):
    results = runs[label]["trace"][-1]["results"]
    print(label, "→ 호출별 Tool status:", [(r["tool_name"], r["status"]) for r in results])
# ToolInput 자체는 실행 기록에 남기지 않으므로 같은 질문을 기록용 래퍼로 한 번 더 실행해 확인한다.
seen = []
def recording(name):
    inner = REGISTRY[name]
    def call(tool_input):
        seen.append((name, tool_input.get("context_entity_ids", []), tool_input["task"][-90:]))
        return inner(tool_input)
    return call
rec_registry = {name: recording(name) for name in REGISTRY}
run_main(runs["B. 보조 의도(질병) → 2회 → 충분 종료"]["query"], deps, rec_registry)
display(pd.DataFrame(seen, columns=["Tool", "context_entity_ids", "task(끝부분)"]))

B. 보조 의도(질병) → 2회 → 충분 종료 → 호출별 Tool status: [('action_search', 'OK'), ('disease_search', 'OK')]
C. 법령(샘플) 부분 근거 → 학교 행동 추가 → 2회 종료 → 호출별 Tool status: [('legal_search', 'PARTIAL'), ('action_search', 'OK')]


,Tool,context_entity_ids,task(끝부분)
0,action_search,[],"학교가 수행하는 행동, 역할, 단계 및 근거 찾기"
1,disease_search,"[Disease:MEASLES, ActionConcept:EXCLUDE_FROM_SCHOOL, ActionConcept:REQUEST_MEDICAL_VISIT, ActionConcept:DE...",질병별 사실(잠복기·전파·증상·관리 기준)과 원문 근거 찾기 (앞선 호출 ['action_search']에서 남은 부분: 질문의 일부(DISEASE))


### 사용자에게 보이는 답변

In [5]:
for label, r in runs.items():
    print(f"=== {label} | {r['out']['tools_used']} | {r['out']['status']}")
    print(r["out"]["answer"])
    print()

=== A. 근거 충분 → 1회 종료 | ['action_search'] | ANSWERED
원문에서 확인한 내용입니다.

■ 담임교사
- 감염병 유증상자를 발견한다.  (경기도 p.166 · 대응 1단계) [1]
- 의료기관 진료 결과를 확인하고 조치한다.  (경기도 p.166 · 대응 1단계) [1]
- 결석·조퇴·지각 사유를 확인하고 의심환자를 파악한다.  (교육부 p.43(시나리오 예시)) [5]

■ 보건교사
- 환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.  (교육부 p.36(시나리오 예시) · 대응 1단계) [2]
- 감염병을 의심할 수 있는 증상인지 확인한다.  (경기도 p.167 · 대응 1단계) [3]
- 일일 보건실 이용 학생의 감염병 증상 여부를 확인한다. (시점: 일일)  (교육부 p.42(시나리오 예시)) [4]

■ 담당이 원문에 적혀 있지 않은 행동
- 감염병(의심) 환자 발생 여부를 확인한다. (조건: 결과를 확인 후)  (경기도 p.164 · 대응 1단계) [6]
- 학생을 최대한 안정시킨 상태에서 증상 변화를 주의 깊게 관찰한다.  (교육부 p.56 · 대응 1단계) [7]

■ 원문 그대로 (행동으로 정리되지 않은 문단·표)
- “감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지 이동 시에는 담임(담당) 교사가 동행하며 2m 정도의 거리 유지 담당 교사는 담임교사 권장 (만약 임신부 등 감염병 고위험군인 경우 다른 교사로 대체)”  (교육부 p.54 · 대응 1단계) [8]
- “7) 교사 및 학부모의 역할 ∙ 정기적인 예방접종 권장 ∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 교육(지원)청에 보고 및 조치함”  (교육부 p.197) [9]

그 밖의 근거 2건은 findings·citations에 있습니다.

참고: 원문 인용 Finding은 검색된 원문을 그대로 옮긴 것이다. 해석·적용 판단은 포함하지 않는다. / 역할은 원문이 제시한 예시이며 학교 상황에 따라 조정

## 2. 실DB GraphPath 1건 검증 (CrossDoc 결과)

In [6]:
results = runs["E. 문서 비교(CrossDoc, GraphPath) → 1회 종료"]["trace"][-1]["results"]
paths = [p for r in results for p in r["graph_paths"]]
problems = check_graph_paths(paths, ALLOWED, DRIVER, DATABASE)
print("CrossDoc GraphPath 수:", len(paths), "/ 실DB 대조 실패 경로:", len(problems))
# 발표 증빙은 manifest 선정 범위 안 청크의 경로로 고른다. 범위 밖(경기도 p.54·56 예전 청크)을 인용한 경로 수도 함께 보고한다.
from rag.school.capabilities import active_pages, in_scope
from rag.school.action_tool import page_of
scope = active_pages()
def path_in_scope(p):
    chunks = [e["target_ref"].split(":", 1)[1] for e in p["edges"] if e["target_ref"].startswith("Chunk:")]
    return chunks and all(in_scope(c.split(":")[0], page_of(c), scope) for c in chunks)
outside = [p for p in paths if not path_in_scope(p)]
print("선정 범위 밖 청크를 쓰는 경로:", len(outside), "(CrossDoc 담당자 공유 사항)")
path = next(p for p in paths if path_in_scope(p))
display(pd.DataFrame(path["edges"]))
# 경로의 각 관계를 Neo4j에서 직접 다시 확인(저장 방향 그대로)
LABEL_ID = {"SchoolAction": "action_id", "Chunk": "chunk_id", "Document": "document_id", "SchoolRole": "role_id",
            "ActionConcept": "concept_id", "Disease": "disease_id"}
checks = []
for edge in path["edges"]:
    (sl, sid), (tl, tid) = (edge["source_ref"].split(":", 1), edge["target_ref"].split(":", 1))
    rec, _, _ = DRIVER.execute_query(
        f"MATCH (s:{sl} {{{LABEL_ID[sl]}: $s}})-[r:{edge['relationship_type']}]->(t:{tl} {{{LABEL_ID[tl]}: $t}}) "
        "RETURN count(r) AS n", database_=DATABASE, s=sid, t=tid)
    checks.append({**edge, "Neo4j에 존재": rec[0]["n"] > 0})
display(pd.DataFrame(checks))

CrossDoc GraphPath 수: 11 / 실DB 대조 실패 경로: 0
선정 범위 밖 청크를 쓰는 경로: 2 (CrossDoc 담당자 공유 사항)


,source_ref,relationship_type,target_ref
0,SchoolAction:gyeonggi_school_guide_2025:p0164:a07,INSTANCE_OF,ActionConcept:REQUEST_MEDICAL_VISIT
1,SchoolAction:gyeonggi_school_guide_2025:p0164:a07,SUPPORTED_BY,Chunk:gyeonggi_school_guide_2025:p0164:t01r05
2,Document:gyeonggi_school_guide_2025,HAS_CHUNK,Chunk:gyeonggi_school_guide_2025:p0164:t01r05


,source_ref,relationship_type,target_ref,Neo4j에 존재
0,SchoolAction:gyeonggi_school_guide_2025:p0164:a07,INSTANCE_OF,ActionConcept:REQUEST_MEDICAL_VISIT,True
1,SchoolAction:gyeonggi_school_guide_2025:p0164:a07,SUPPORTED_BY,Chunk:gyeonggi_school_guide_2025:p0164:t01r05,True
2,Document:gyeonggi_school_guide_2025,HAS_CHUNK,Chunk:gyeonggi_school_guide_2025:p0164:t01r05,True


## 3. M4 판정

In [7]:
s = {label[:1]: r["out"] for label, r in runs.items()}
assert s["A"]["search_rounds"] == 1 and s["A"]["status"] == "ANSWERED"                       # 충분하면 추가 호출 없음
assert s["B"]["tools_used"] == ["action_search", "disease_search"] and s["B"]["status"] == "ANSWERED"
assert s["C"]["search_rounds"] == 2 and s["D"]["search_rounds"] == 2 and s["D"]["status"] == "PARTIAL"
assert all(r["out"]["search_rounds"] <= MAX_TOOL_CALLS for r in runs.values())
assert all(r["db"] in (None, "통과") for r in runs.values())
assert not problems and all(c["Neo4j에 존재"] for c in checks)
print("M4: 1-Tool 종료(A·E), 추가 Tool 호출 후 정상 종료(B·C·D), 실DB GraphPath 검증(E) — 통과")
out_path = ROOT / "data/interim/school/m4_loop_logs_20261106.json"
out_path.write_text(json.dumps({label: {k: r[k] for k in ("query", "out", "db", "seconds")} | {
    "trace": [t for t in r["trace"] if t["step"] != "tool_results"]} for label, r in runs.items()},
    ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("저장:", out_path.relative_to(ROOT))

M4: 1-Tool 종료(A·E), 추가 Tool 호출 후 정상 종료(B·C·D), 실DB GraphPath 검증(E) — 통과
저장: data\interim\school\m4_loop_logs_20261106.json


In [8]:
DRIVER.close()